# Gated QGFD — staged experiments (one free Kaggle T4)

**The claim being tested.** Fixed-α QGFD lost: across 3 models × 3 seeds × 4 noise
rates its *absolute* perplexity under noise was lower in **0 of 36** measurements, and
the published-looking robustness gap was a denominator artefact of a relative metric.
The diagnosis is that at α = 0 QGFD *is* softmax, so a fixed α > 0 is a constant
perturbation of a converged optimum and `ΔL ≈ ½δᵀHδ = O(α²) ≥ 0` in every direction.
The fix under test here is a **learned, zero-initialised, per-head, per-position gate**:

```
p          = (1 − g)·p⁰ + g·(p⁰P),        P = softmax(KKᵀ/√d)
g_{h,t}    = g_max · tanh( a_h·Ĥ(p⁰)_t + b_h·margin(p⁰)_t + r_h ),   a = b = r = 0 at init
```

> *Some attention heads benefit from learned, selective diffusion when the input or
> context is unreliable, while precision-retrieval heads should remain ordinary softmax.*

| Stage | What runs | Time | Gate to the next stage |
|---|---|---|---|
| Preflight | α = 0 exactness, gate-is-zero, patch verify | 30 min | all three PASS |
| **0** | per-head α-gradients, per-layer α oracle, corruption screen — **no training** | 40 min | `proceed` / `pivot` / `stop` |
| **1** | gate only, base frozen, ~1k parameters, 3 seeds | 1.5 h | ε cleared **and** beats shuffled-P |
| **2** | trust-aware top-k graph + Q/K LoRA, 2 arms | 3 h | paired noisy-CE win, CI excludes 0 |
| **3** | Qwen2.5-0.5B + TinyLlama-1.1B, recipe transferred un-tuned | 2 h | — |

Under 8 GPU-hours end to end; **70 minutes if Stage 0 says stop**, which is the point of
running Stage 0 first. Every stage writes its own JSON, so the run is coherent if you
stop after any cell.

Primary metric throughout: **paired per-window cross-entropy in nats** on byte-identical
corrupted text. Never a relative delta of perplexities — that statistic is what produced
the phantom +12.6 pp.

Full write-up: `docs/gated-qgfd-experiments.md`. Implementation: `scripts/gated_experiments.py`.

---

## 1 · Environment

In [ ]:
# Kaggle: turn Internet ON (Settings -> Internet) before running this, or the clone,
# the pip installs and the WikiText-2 download all fail. Accelerator: GPU T4 x2.
import os, subprocess, sys

REPO_URL, REPO_DIR = "https://github.com/rajboopathiking/TorchDire.git", "TorchDire"
if not os.path.isdir("torchdire"):                 # not already inside the repo
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.40", "peft>=0.10", "datasets>=2.18",
                "accelerate", "matplotlib"], check=False)

In [ ]:
# --- GPU and dtype -----------------------------------------------------------
# fp16, NOT bf16, and this is a real ~2x on this hardware. bf16 *tensor cores* need
# compute capability >= 8.0; a T4 is sm_75, so torch emulates bf16 there -- it is
# correct but slower, and torch.cuda.is_bf16_supported() still returns True, which
# is how the earlier 9-hour run ended up in emulated bf16 without anyone noticing.
# The operator keeps p0, P and the loss in fp32 regardless, which is where the
# precision actually matters.
import torch

DTYPE_OVERRIDE = None      # None = auto (fp16 on GPU); or pin "bfloat16"/"float32"

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    DTYPE = DTYPE_OVERRIDE or "float16"
    print(f"{p.name} | sm_{p.major}{p.minor} | {p.total_memory / 2**30:.1f} GiB "
          f"| {torch.cuda.device_count()} device(s)")
    print(f"bf16 tensor cores (sm>=80): {p.major >= 8} | "
          f"torch.cuda.is_bf16_supported(): {torch.cuda.is_bf16_supported()}")
    if p.major < 8:
        print("  -> pre-Ampere: bf16 would be EMULATED. Running fp16.")
    if torch.cuda.device_count() > 1:
        print("  -> 2 devices available. This notebook uses cuda:0 only; to halve "
              "wall-clock, run a second copy with CUDA_VISIBLE_DEVICES=1 on the "
              "other seed set (the arms are independent by construction).")
else:
    DTYPE = DTYPE_OVERRIDE or "float32"
    print("No CUDA device -- CPU/float32. Set QUICK = True below.")
print("torch", torch.__version__, "| dtype for this run:", DTYPE)

## 2 · Configuration

Defaults are already the T4-sized ones; the only knobs worth touching on a first run
are `QUICK` and `seeds`.

- `seq_len=128` — eager attention is O(n²) materialised **and** the diffusion product
  is O(n³), and we pay both because the operator must see the probability matrix, so no
  FlashAttention. This is the biggest lever after dtype.
- `g_max=0.05`, `signed=True` — the tanh gate is exactly 0 at init with a live gradient.
  A one-sided sigmoid gate cannot be both.
- `train_corruption="word_drop"` — tokenisation-preserving. `char` noise re-segments
  words into subword fragments, so a win there is confounded with fragmentation repair.
- `epsilon_clean_nats=0.003` (≈0.3% relative PPL) — the clean-CE regression budget.

`QUICK=True` swaps in `llama-160m` at 4 steps: a few minutes, exercises every code path,
and every number it prints is meaningless. Run it once, then set it back.

In [ ]:
# --- Configuration -----------------------------------------------------------
from dataclasses import replace

from scripts.gated_experiments import GatedConfig, LARGER_MODELS, apply_quick

QUICK = False     # True => llama-160m, 4 steps: code-path rehearsal, numbers are noise
FORCE = False     # True => run stages 1-3 even if the stop rule fires. Report it if you do.

CFG = GatedConfig(
    model_id="HuggingFaceTB/SmolLM2-135M",   # llama, 135M -- the development model
    dtype=DTYPE, device="auto", seq_len=128,
    g_max=0.05, signed=True,                 # tanh gate, sharpening reachable
    top_k=8, self_loop=1.0, reliability=True,  # Stage 2 trust graph
    max_steps=300, grad_accum=4,
    train_corruption="word_drop", eval_corruption="word_drop", eval_rate=0.10,
    epsilon_clean_nats=0.003,
    seeds=(0, 1, 2),
    out_dir="qgfd_gated_results",
)
if QUICK:
    CFG = apply_quick(replace(CFG, out_dir="qgfd_gated_results_quick"))

print(f"model      {CFG.model_id}")
print(f"dtype      {CFG.dtype}   seq_len {CFG.seq_len}")
print(f"gate       g_max {CFG.g_max}  signed {CFG.signed}  "
      f"entropy {CFG.use_entropy}  margin {CFG.use_margin}")
print(f"train      {CFG.max_steps} steps x accum {CFG.grad_accum}  "
      f"gate_lr {CFG.gate_lr}  gamma_l1 {CFG.gamma_l1}")
print(f"corruption train {CFG.train_corruption} {CFG.curriculum} -> "
      f"eval {CFG.eval_corruption} @ {CFG.eval_rate}")
print(f"epsilon    {CFG.epsilon_clean_nats} nats   seeds {CFG.seeds}")
print(f"out_dir    {CFG.out_dir}   QUICK {QUICK}  FORCE {FORCE}")

## 3 · Preflight — three things that would silently invalidate everything

1. **α = 0 must be bit-exact softmax.** This is contribution (1) of the paper and it is
   a falsifiable claim about the implementation. If it fails, nothing downstream should
   be believed.
2. **The gate must be exactly 0 at init *and* have a live gradient there.** Exactly 0
   means training starts at the optimum the pretrained model already found; a live
   gradient means it can leave. `tanh` gives both; `g_max·σ(z)` gives neither at once.
3. **The patch must actually be reached.** `verify_patch()` raises rather than reporting
   a number if the operator silently no-ops — which is what happens on GPT-2 / OPT /
   GPT-Neo, since only Llama / Mistral / Qwen2 are genuinely wired.

In [ ]:
# --- Preflight ---------------------------------------------------------------
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

from torchdire import QGFDOperator, SoftmaxOperator, wrap_model_with_qgfd_operator
from scripts.gated_experiments import GatedQGFDOperator, install_per_layer

PREFLIGHT = {}

# (1) alpha = 0 is bit-for-bit softmax
_tok = AutoTokenizer.from_pretrained(CFG.model_id)
_ids = _tok("Diffusion over the key graph must vanish when alpha is zero.",
            return_tensors="pt")["input_ids"]
_lg = {}
for _name, _op in (("softmax", SoftmaxOperator()),
                   ("qgfd_alpha0", QGFDOperator(diffusion_steps=1, target_alpha=0.0,
                                                warmup_steps=0, is_causal=True))):
    _m = AutoModelForCausalLM.from_pretrained(CFG.model_id, torch_dtype=torch.float32)
    _m = wrap_model_with_qgfd_operator(_m, _op, verbose=False).eval()
    with torch.no_grad():
        _lg[_name] = _m(_ids).logits.float()
    del _m
_dev = (_lg["softmax"] - _lg["qgfd_alpha0"]).abs().max().item()
PREFLIGHT["alpha0_exact"] = _dev == 0.0
print(f"[1] alpha=0 max |delta logit| = {_dev:.3e}  "
      f"{'PASS' if PREFLIGHT['alpha0_exact'] else 'FAIL'}")

# (2) the gate is exactly zero at init, with a live gradient
_op = GatedQGFDOperator(num_heads=4, g_max=CFG.g_max, signed=CFG.signed)
_s, _k = torch.randn(2, 4, 16, 16), torch.randn(2, 4, 16, 8)
_causal = torch.tril(torch.ones(16, 16, dtype=torch.bool))
_mask = torch.zeros(1, 1, 16, 16).masked_fill(~_causal[None, None], -1e9)
with torch.no_grad():
    PREFLIGHT["gate_zero_exact"] = bool(
        torch.equal(_op(_s, _k, _mask), F.softmax(_s + _mask, dim=-1)))
# `p` sums to 1 by construction, so d(p.sum())/d(gate) vanishes for reasons that
# have nothing to do with trainability. Probe a random linear functional instead.
torch.manual_seed(0)
(_op(_s, _k, _mask) * torch.randn(2, 4, 16, 16)).sum().backward()
_g = max(float(t.grad.abs().max())
         for t in (_op.gate_bias, _op.w_entropy, _op.w_margin))
PREFLIGHT["gate_grad_alive"] = _g > 0.0
print(f"[2] gate output == softmax at init: "
      f"{'PASS' if PREFLIGHT['gate_zero_exact'] else 'FAIL'}   "
      f"max |dL/dgate| = {_g:.3e} "
      f"{'PASS' if PREFLIGHT['gate_grad_alive'] else 'FAIL (untrainable)'}")

# (3) the patch is reached -- install_per_layer calls verify_patch, which raises
_m = AutoModelForCausalLM.from_pretrained(CFG.model_id, torch_dtype=torch.float32)
try:
    _ops = install_per_layer(
        _m, lambda i, h: GatedQGFDOperator(num_heads=h, g_max=CFG.g_max),
        device="cpu", verify_tok=_tok, label="preflight")
    PREFLIGHT["patch_live"] = True
    print(f"[3] patch verified on {len(_ops)} layers "
          f"({sum(p.numel() for o in _ops for p in o.parameters())} gate params) PASS")
except Exception as _e:
    PREFLIGHT["patch_live"] = False
    print(f"[3] FAIL: {type(_e).__name__}: {_e}")
del _m

print("\npreflight:", "ALL PASS" if all(PREFLIGHT.values()) else f"FAILURES {PREFLIGHT}")

## 4 · Stage 0 — does the headroom exist at all? (~40 min, no training)

A zero-initialised gate can only exploit heads where diffusion *already* reduces noisy
loss at the margin. That is directly measurable without writing a gate.

| Probe | Method |
|---|---|
| **0a** per-head α-gradient | `learnable_alpha=True` exposes a per-head `alpha_param`; one forward+backward at α ≈ 0 yields `∂CE_noisy/∂α_h` for **every head in every layer at once** |
| **0b** per-layer α oracle | α = 0.05 on exactly one layer, 0 elsewhere; absolute paired CE. Confirms 0a at the real α and catches second-order rescues |
| **0c** corruption screen | fixed α, absolute paired CE, one row per corruption family |

Reading 0a: `∂CE_noisy/∂α_h < 0` means diffusion on that head lowers noisy loss at the
margin — real headroom. `> 0` means the optimiser wants α **negative**, i.e. to
*sharpen* — which inverts the mechanism claim and is a temperature change that folds
into `W_Q` for free.

**The stop rule was fixed before any of these numbers were seen**, because a stop rule
that gets overridden is decoration:

| Outcome | Decision |
|---|---|
| no head with a negative noisy gradient, no layer helps, no family wins | **stop** — publish the negative result plus the metric critique. A zero-init gate cannot reach states a fixed α did not already probe. |
| gradients uniformly positive | **pivot** to the sharpening / temperature story |
| some heads or some family show headroom | **proceed**, restricting the Stage 1 gate to the layers 0b flagged |

In [ ]:
# --- Stage 0 -----------------------------------------------------------------
import json, time

from scripts.gated_experiments import stage0

_t0 = time.time()
S0 = stage0(CFG)
print(f"\nstage 0 wall-clock: {(time.time() - _t0) / 60:.1f} min")

In [ ]:
# --- Stage 0 readout ---------------------------------------------------------
_a, _b, _c = S0["s0a"], S0["s0b"], S0["s0c"]

print("0a  per-head dCE_noisy/dalpha  (negative = diffusion helps at the margin)")
_sa = _a["summary"]
print(f"    {_sa['n_heads_negative_noisy']}/{_sa['n_heads_total']} heads negative "
      f"({_sa['frac_negative_noisy']:.1%})   mean {_sa['mean_grad_noisy']:+.3e}   "
      f"min {_sa['min_grad_noisy']:+.3e}   verdict: {_a['verdict']}")
print("    layer  mean dCE_noisy/dalpha")
for _i, _v in enumerate(_a["grad_noisy_mean_by_layer"]):
    print(f"    {_i:>5}  {_v:+.4e}")
print("    most promising heads (helps under noise, cheap when clean):")
for _r in _sa["top_heads"][:8]:
    print(f"      L{_r['layer']:>2} H{_r['head']:>2}  noisy {_r['d_noisy']:+.3e}  "
          f"clean {_r['d_clean']:+.3e}")

print("\n0b  alpha=%.2f on one layer only, absolute paired CE vs alpha=0" % CFG.probe_alpha)
print(f"    baseline noisy CE {_b['baseline_noisy_ce']:.4f} nats")
print("    layer   delta_CE      ci95     sig")
for _r in _b["by_layer"]:
    _d = "     n/a" if _r["delta_ce"] is None else f"{_r['delta_ce']:+.5f}"
    _ci = "  n/a" if _r["ci95"] is None else f"{_r['ci95']:.5f}"
    print(f"    {_r['layer']:>5}  {_d}  {_ci}   {'*' if _r['sig'] else ' '}")
print(f"    helpful layers: {_b['summary']['helpful_layers'] or 'none'}   "
      f"verdict: {_b['verdict']}")

print("\n0c  corruption screen (delta_CE = QGFD - softmax, negative = QGFD better)")
print("    family           tok?  softmax_CE  qgfd_CE   delta_CE   ci95     win%  sig")
for _r in _c["by_corruption"]:
    _d = "    n/a" if _r["delta_ce"] is None else f"{_r['delta_ce']:+.5f}"
    _ci = "  n/a" if _r["ci95"] is None else f"{_r['ci95']:.5f}"
    _w = " n/a" if _r["win_frac"] is None else f"{_r['win_frac']:.0%}"
    print(f"    {_r['corruption']:<15} {'keep' if _r['preserves_tokenisation'] else 'BRK '} "
          f"  {_r['softmax_ce']:>9.4f} {_r['qgfd_ce']:>9.4f}  {_d}  {_ci}  {_w:>5}  "
          f"{'*' if _r['sig'] else ''}")
print(f"    winning families: {_c['summary']['winning_families'] or 'none'}   "
      f"(tokenisation-preserving: "
      f"{_c['summary']['token_preserving_wins'] or 'none'})")

DECISION = S0["decision"]["decision"]
LAYERS = _b["summary"]["helpful_layers"] or None
print("\n" + "=" * 78)
print(f"STAGE 0 DECISION: {DECISION.upper()}")
print(S0["decision"]["why"])
if LAYERS:
    print(f"Stage 1 gate restricted to layers {LAYERS}.")
print("=" * 78)

## 5 · Stage 1 — gate only, base model frozen (~1.5 h)

The only trainable parameters are 3 per head (≈1k total); every base weight is frozen.

```
L = CE_clean + λ_n(t)·CE_noisy + γ·mean|g|
```

λ_n ramps from 0 over the first 30% of steps and the corruption rate follows the
2% → 8% curriculum, so the gate first learns not to break clean text and only then
learns when to fire. The L1 term on |g| is the "stay at softmax unless you earn it"
pressure, and it is symmetric — it penalises sharpening and smoothing equally.

Nothing here can overfit in an interesting way, which is the point: if the gate cannot
win with the base weights untouched, a win after LoRA is a win by the adapters.

The softmax reference is **the same weights with the gate switched off**, so the
comparison is exactly paired: one model, one set of windows, one difference.

**Success = constrained Pareto improvement:** clean ΔCE ≤ ε **and** noisy ΔCE < 0 with
the paired CI excluding zero — *and* the trained gate must fail to reproduce its gain
with `uniform` or `shuffled` P. If a structure-free P works just as well, the gate
learned an adaptive temperature schedule (free, folds into `W_Q`), not graph routing,
and it gets reported that way.

In [ ]:
# --- Stage 1 -----------------------------------------------------------------
from scripts.gated_experiments import stage1

S1 = None
if DECISION == "proceed" or FORCE:
    if DECISION != "proceed":
        print(f"!! FORCE=True: running Stage 1 past a '{DECISION}' verdict. "
              f"Report this in the paper.\n")
    _t0 = time.time()
    S1 = stage1(CFG, layers=LAYERS)
    print(f"\nstage 1 wall-clock: {(time.time() - _t0) / 60:.1f} min")
else:
    print(f"Stage 0 said '{DECISION}'. Stopping here, as pre-declared.\n"
          f"{S0['decision']['why']}\n\n"
          f"The deliverable is now the negative result plus the relative-metric "
          f"critique. Skip to the final cell.")

In [ ]:
# --- Stage 1 readout ---------------------------------------------------------
def _row(d):
    if d is None or d["mean"] is None:
        return "n/a"
    return (f"{d['mean']:+.5f} +/- {d['ci95']:.5f} nats  "
            f"(n={d['n']}, win {d['win_frac']:.0%}) {'*' if d['sig'] else ' '}")


if S1:
    print("seed  clean delta_CE                          noisy delta_CE"
          "                          pareto  controls")
    for _r in S1["runs"]:
        print(f"{_r['seed']:>4}  {_row(_r['delta_ce_clean']):<38}"
              f"{_row(_r['delta_ce_noisy']):<38}"
              f"{str(_r['pareto']):<7} {(_r.get('controls') or {}).get('verdict', '-')}")
    _s = S1["summary"]
    print(f"\nmean clean  {_s['mean_delta_ce_clean']:+.5f} nats   "
          f"(epsilon budget {CFG.epsilon_clean_nats})")
    print(f"mean noisy  {_s['mean_delta_ce_noisy']:+.5f} nats")
    print(f"pareto      {_s['n_pareto']}/{_s['n_seeds']} seeds")
    print(f"controls    {_s['controls']}  -> graph needed on all seeds: "
          f"{_s['graph_matters_all_seeds']}")
    print(f"VERDICT     {_s['verdict'].upper()}")
    if _s["verdict"] == "temperature":
        print("  A structure-free P reproduced the gain. This is an adaptive "
              "temperature\n  schedule, not graph routing -- report it as that, and "
              "note that it folds\n  into W_Q at zero inference cost.")
    for _r in S1["runs"]:
        _auc = _r.get("gate_auc_noisy_vs_clean")
        if _auc is not None:
            print(f"  seed {_r['seed']}: gate AUC (corrupted vs clean windows) "
                  f"{_auc:.3f}"
                  + ("  <- high: this is a noise detector + smoother" if _auc > 0.8
                     else ""))

## 6 · Stage 2 — trust-aware sparse graph + Q/K LoRA (~3 h)

Runs only if Stage 1 cleared ε **and** beat the shuffled-P control.

```
P_ij = softmax_j( k̂_i·k̂_j/τ + s·[i=j] + λ·(−z_j) ),   restricted to top-k in the causal prefix
```

- **top-k (k=8) is not optional** — it takes `p⁰P` from O(n³) to O(n²k), which is what
  makes this stage affordable on a T4 at all.
- **Causal mask and top-k are applied to the logits, before normalisation**, so every
  row is a proper distribution over exactly the surviving edges.
- **Reliability is a free signal, not a learned sub-model.** `z_j` is the standardised
  key norm; unusually long keys dominate similarity scores for reasons unrelated to
  content, so `−z_j` is a defensible prior. λ is one scalar per layer, initialised to 0,
  so the term starts inert.
- **LoRA on `q_proj`/`k_proj` only** — not q/k/v/o. If the hypothesis is about the key
  graph, V and O adapters let the model win for unrelated reasons and the attribution
  is gone.
- `agreement_ij` (cross-layer attention agreement) is **deferred**: it is the most
  expensive feature in the design and the least justified before the cheap ones show
  something.

Once LoRA moves the weights, "gate off" is no longer the right control, so this stage
trains **two arms** — LoRA-only vs gate+trust+LoRA — on identical seed, data, steps and
LR, and pairs them at the window level.

In [ ]:
# --- Stage 2 -----------------------------------------------------------------
from scripts.gated_experiments import stage2

S2 = None
_s1_ok = bool(S1 and S1["summary"]["verdict"] == "success")
if _s1_ok or (FORCE and S1):
    if not _s1_ok:
        print("!! FORCE=True: running Stage 2 past a non-success Stage 1.\n")
    _t0 = time.time()
    S2 = stage2(CFG, layers=LAYERS)
    print(f"\nstage 2 wall-clock: {(time.time() - _t0) / 60:.1f} min")
elif S1:
    print(f"Stage 1 verdict '{S1['summary']['verdict']}' -- not clearing Stage 2. "
          f"A gate that\ncannot win with the base model frozen cannot be credited "
          f"for a win after LoRA.")
else:
    print("Stage 1 did not run.")

In [ ]:
# --- Stage 2 readout ---------------------------------------------------------
if S2:
    print("gate+trust+LoRA  minus  LoRA-only, paired at the window level")
    print("seed  clean delta_CE                          noisy delta_CE"
          "                          pareto")
    for _r in S2["runs"]:
        print(f"{_r['seed']:>4}  {_row(_r['delta_ce_clean']):<38}"
              f"{_row(_r['delta_ce_noisy']):<38}{_r['pareto']}")
    _s = S2["summary"]
    print(f"\nmean clean {_s['mean_delta_ce_clean']:+.5f}   "
          f"mean noisy {_s['mean_delta_ce_noisy']:+.5f}   "
          f"pareto {_s['n_pareto']}/{_s['n_seeds']}   "
          f"VERDICT {_s['verdict'].upper()}")
    for _r in S2["runs"]:
        for _arm, _v in _r["arms"].items():
            _g = _v.get("gate")
            _m = None if not _g else _g.get("mean_absmean")
            print(f"  seed {_r['seed']} {_arm:<16} train {_v['train_s'] / 60:.1f} min  "
                  f"lora params {_v['n_lora_params']}  "
                  f"mean |g| {'n/a' if _m is None else f'{_m:.5f}'}")

## 7 · Stage 3 — does the recipe transfer? (~2 h)

Qwen2.5-0.5B (cross-family: qwen2 adapter, not llama) and TinyLlama-1.1B (real GQA,
32 heads / 4 KV), one seed each, hyper-parameters transferred **as-is**.

Deliberately un-tuned. If the effect is a mechanism it survives the transfer; if it
needs its hyper-parameters refitted per checkpoint it is a fit, and the honest thing is
to report it as one. Note the prior: on the fixed-α experiments the headline gap
*reversed sign* on TinyLlama-1.1B, so this is the stage most likely to kill the result.

In [ ]:
# --- Stage 3 -----------------------------------------------------------------
from scripts.gated_experiments import stage3

S3 = None
_s2_ok = bool(S2 and S2["summary"]["verdict"] == "success")
if _s2_ok or (FORCE and S1):
    _t0 = time.time()
    S3 = stage3(CFG, models=LARGER_MODELS, with_lora=_s2_ok)
    print(f"\nstage 3 wall-clock: {(time.time() - _t0) / 60:.1f} min")
    for _mid, _v in S3["by_model"].items():
        if "error" in _v:
            print(f"  {_mid:<42} ERROR {_v['error']}")
        else:
            _s = _v["summary"]
            print(f"  {_mid:<42} clean {_s['mean_delta_ce_clean']:+.5f}  "
                  f"noisy {_s['mean_delta_ce_noisy']:+.5f}  "
                  f"{_s['verdict'].upper()}")
    print(f"\ntransfers cleanly: {S3['summary']['transfers']}")
else:
    print("Stage 2 did not clear -- no recipe to transfer.")

## 8 · Diffusion-utilisation table

All of this comes out of the forward passes that already ran, so it is free. It is also
what turns a number into a mechanism claim — or exposes that there isn't one.

- **mean signed gate per layer** — where the model chose to diffuse, and in which
  direction. Negative means the head asked to *sharpen*.
- **fraction of heads with |g| ≈ 0** — the "stayed softmax" count. If the hypothesis is
  right this should be large: selectivity is the whole thesis.
- **gate AUC, corrupted vs clean windows** — if this is high, what was built is a noise
  detector plus a smoother, and it should be described that way rather than as graph
  routing.

In [ ]:
# --- Gate telemetry ----------------------------------------------------------
def gate_table(res, label):
    for _key in ("gate_clean", "gate_noisy"):
        _g = res.get(_key)
        if not _g or not _g.get("per_layer_per_head"):
            continue
        _per_layer = [sum(h) / len(h) for h in _g["per_layer_per_head"]]
        _inert = _g.get("frac_inert_by_layer") or []
        print(f"\n{label} / {_key}  (mean |g| over windows: "
              f"{_g.get('mean_absmean')})")
        print("  layer   mean signed g   frac |g|~0")
        for _i, _v in enumerate(_per_layer):
            _f = "n/a" if _i >= len(_inert) or _inert[_i] is None \
                else f"{_inert[_i]:.2f}"
            print(f"  {_i:>5}   {_v:+.6f}      {_f}")


if S1:
    for _r in S1["runs"][:1]:            # one seed is enough for the shape
        gate_table(_r, f"stage1 seed {_r['seed']}")
        _auc = _r.get("gate_auc_noisy_vs_clean")
        print(f"\n  gate AUC (corrupted vs clean): "
              f"{'n/a' if _auc is None else f'{_auc:.3f}'}")
        _flat = [h for layer in (_r.get('gate_noisy') or {}).get(
            'per_layer_per_head', []) for h in layer]
        if _flat:
            _n0 = sum(1 for v in _flat if abs(v) < 1e-4)
            print(f"  heads that stayed softmax (|g| < 1e-4): "
                  f"{_n0}/{len(_flat)} ({_n0 / len(_flat):.0%})")
            print(f"  heads that chose to SHARPEN (g < 0): "
                  f"{sum(1 for v in _flat if v < -1e-4)}/{len(_flat)}")

In [ ]:
# --- Optional: per-head gate heat map ----------------------------------------
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:                       # plot is a nicety, not a result
    plt = None
    print("matplotlib not installed -- skipping the heat map.")

if plt and S1 and S1["runs"]:
    _g = (S1["runs"][0].get("gate_noisy") or {}).get("per_layer_per_head")
    if _g:
        _m = torch.tensor(_g)
        _lim = float(_m.abs().max()) or 1e-6
        fig, ax = plt.subplots(figsize=(8, 0.4 * len(_g) + 1.5))
        im = ax.imshow(_m, cmap="coolwarm", vmin=-_lim, vmax=_lim, aspect="auto")
        ax.set_xlabel("head")
        ax.set_ylabel("layer (gated layers only)")
        ax.set_title("mean signed gate on corrupted input\n"
                     "red = diffuse, blue = sharpen, white = stayed softmax")
        fig.colorbar(im, ax=ax, label="g")
        fig.tight_layout()
        _p = os.path.join(CFG.out_dir, "gate_heatmap.png")
        fig.savefig(_p, dpi=140)
        print("wrote", _p)
        plt.show()

## 9 · What to report

The result is one of four things, and all four are publishable. What is *not*
publishable is a relative-degradation number.

In [ ]:
# --- Summary -----------------------------------------------------------------
SUMMARY = {
    "config": {"model_id": CFG.model_id, "dtype": CFG.dtype,
               "seq_len": CFG.seq_len, "g_max": CFG.g_max,
               "train_corruption": CFG.train_corruption,
               "eval_corruption": CFG.eval_corruption,
               "eval_rate": CFG.eval_rate, "seeds": list(CFG.seeds),
               "quick": QUICK, "forced": FORCE},
    "preflight": PREFLIGHT,
    "stage0": {"decision": DECISION, "why": S0["decision"]["why"],
               "helpful_layers": LAYERS,
               "heads_negative": S0["s0a"]["summary"]["n_heads_negative_noisy"],
               "heads_total": S0["s0a"]["summary"]["n_heads_total"],
               "winning_families": S0["s0c"]["summary"]["winning_families"]},
    "stage1": S1["summary"] if S1 else None,
    "stage2": S2["summary"] if S2 else None,
    "stage3": S3["summary"] if S3 else None,
}
_path = os.path.join(CFG.out_dir, "notebook_summary.json")
os.makedirs(CFG.out_dir, exist_ok=True)
with open(_path, "w") as fh:
    json.dump(SUMMARY, fh, indent=2, default=str)
print(json.dumps(SUMMARY, indent=2, default=str))
print("\nwrote", _path)

_v1 = (S1 or {}).get("summary", {}).get("verdict")
_v2 = (S2 or {}).get("summary", {}).get("verdict")
if DECISION == "stop":
    _story = ("NEGATIVE RESULT. No first-order headroom anywhere. Publish the 0/36 "
              "absolute-PPL result plus the relative-metric critique; the critique "    
              "generalises to any robustness paper that reports relative "
              "degradation, and the LoRA run demonstrates it with no QGFD in the "
              "comparison at all.")
elif DECISION == "pivot":
    _story = ("INVERTED MECHANISM. Every head wants alpha < 0 -- the model asks to "
              "SHARPEN, not smooth. Sharpening is a temperature change that folds "
              "into W_Q at zero inference cost. Report the temperature control as "
              "the winner, not as the null.")
elif _v1 == "temperature" or _v2 == "temperature":
    _story = ("ADAPTIVE TEMPERATURE, NOT GRAPH ROUTING. A structure-free P "
              "reproduced the gain, so the learned quantity is 'spread mass when "
              "the row is uncertain'. Still a result -- and a cheaper one -- but it "
              "must not be described as diffusion over the key graph.")
elif _v2 == "success" or _v1 == "success":
    _story = ("SELECTIVE DIFFUSION HOLDS. Constrained-Pareto improvement with the "
              "paired CI excluding zero, and the shuffled-P control fails to "
              "reproduce it. Report absolute paired CE in nats, the fraction of "
              "heads that stayed softmax, and the gate AUC -- never a relative "
              "degradation.")
else:
    _story = ("NO EFFECT AT THIS SCALE. Headroom existed at the margin but the gate "
              "could not convert it. Report the Stage 0 headroom map alongside the "
              "null: it is the most informative negative result available here.")
print("\n" + "=" * 78 + f"\n{_story}\n" + "=" * 78)
print("\nContributions that stand regardless: (1) alpha=0 exactness "
      f"[{'verified' if PREFLIGHT.get('alpha0_exact') else 'FAILED'}], "
      "(3) the single-GPU recipe,\nand the relative-metric critique.")
print(f"\nArtefacts in {CFG.out_dir}/: stage0.json, stage1.json, stage2.json, "
      "stage3.json,\nsummary.json, notebook_summary.json, gate_heatmap.png")